In [1]:
import sys

sys.path.append("../..")

In [2]:
# import os
# os.environ['PROJ_LIB'] = '/opt/anaconda3/envs/claymodel/share/proj'

# explicitly set the PROJ data directory to ensure pyproj can resolve EPSG codes correctly.
from pyproj.datadir import set_data_dir

set_data_dir("/opt/anaconda3/envs/claymodel/share/proj")

/opt/anaconda3/envs/claymodel/lib/python3.11/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [3]:
import numpy as np
import pandas as pd
import pystac_client
import stackstac
from rasterio.enums import Resampling

In [ ]:
# point over DD

lat, lon = 30.30622, 78.03225

In [6]:
delta_lat = 10 / 111320  # ≈ 9e-5
delta_lon = 10 / (111320 * np.cos(np.deg2rad(lat)))  # ≈ 1e-4

bbox = (
    lon - delta_lon / 2,
    lat - delta_lat / 2,
    lon + delta_lon / 2,
    lat + delta_lat / 2,
)

In [9]:
# range of date

start = "2024-01-01"
end = "2025-12-05"

In [29]:
# get data from STAC catalouge

stac_api = "https://earth-search.aws.element84.com/v1"
collection = "sentinel-2-l2a"

# Search the catalouge
catalog = pystac_client.Client.open(stac_api)
search = catalog.search(
    collections=[collection],
    datetime=f"{start}/{end}",
    bbox=bbox,
    max_items=1000,
    query={"eo:cloud_cover": {"lt": 50}},  # less than 50% cloud cover,
)

all_items = search.item_collection()

# reduce to one per date(there might be duplicates based on the loc)

items = []
dates = []
for item in all_items:
    if item.datetime.date() not in dates:
        items.append(item)
        dates.append(item.datetime.date())


print(f"Found {len(items)} items")

Found 105 items


In [18]:
import planetary_computer as pc

signed_items = [pc.sign(item) for item in items]

In [25]:
# create a bounding box around pt of interest

# Extract coordinate system from first item
import geopandas as gpd
from shapely import Point

epsg = int(items[0].properties["proj:code"].replace("EPSG:", ""))

# convert point of interest into image projection
# assuming all images are in the same projection

poidf = gpd.GeoDataFrame(
    pd.DataFrame(),
    crs="EPSG:4326",
    geometry=[Point(lon, lat)],
).to_crs(epsg)

coords = poidf.iloc[0].geometry.coords[0]

In [26]:
# create bounds in projection

size = 256
gsd = 10
bounds = (
    coords[0] - (size * gsd) // 2,
    coords[1] - (size * gsd) // 2,
    coords[0] + (size * gsd) // 2,
    coords[1] + (size * gsd) // 2,
)

In [27]:
# Retrieving the imagery data

# retrieving the pixel values, for the bounding box in the target projection.
# ex - here using RGB & NIR bands only

stack1 = stackstac.stack(
    items,
    bounds=bounds,
    snap_bounds=False,
    epsg=epsg,
    resolution=gsd,
    dtype="float64",
    rescale=False,
    fill_value=0,
    assets=["blue", "green", "red", "nir"],
    resampling=Resampling.nearest,
)

print(stack1)

stack1 = stack1.compute()

<xarray.DataArray 'stackstac-658b12a23117c46eb4ccc283c2ab5b6b' (time: 105,
                                                                band: 4,
                                                                y: 256, x: 256)> Size: 220MB
dask.array<fetch_raster_window, shape=(105, 4, 256, 256), dtype=float64, chunksize=(1, 1, 256, 256), chunktype=numpy.ndarray>
Coordinates: (12/52)
  * time                                     (time) datetime64[ns] 840B 2024-...
    id                                       (time) <U24 10kB 'S2A_43RGP_2024...
  * band                                     (band) <U5 80B 'blue' ... 'nir'
  * x                                        (x) float64 2kB 7.903e+05 ... 7....
  * y                                        (y) float64 2kB 3.358e+06 ... 3....
    grid:code                                (time) <U10 4kB 'MGRS-43RGP' ......
    ...                                       ...
    s2:tile_id                               (time) object 840B None ... 'S2C...

RuntimeError: Error opening 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2024/3/S2A_43RGP_20240314_0_L2A/B03.tif': RasterioIOError("'/vsicurl/https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2024/3/S2A_43RGP_20240314_0_L2A/B03.tif' does not exist in the file system, and is not recognized as a supported dataset name.")

In [30]:
import stackstac
from rasterio.enums import Resampling
from dask.diagnostics import ProgressBar


# Wrap stackstac with try/except logging
def safe_stack(items, bounds, epsg, gsd):
    good_items = []
    for item in items:
        try:
            # Try opening one band to check accessibility
            href = item.assets["red"].href
            print(f"Checking item {item.id} -> {href}")
            good_items.append(item)
        except Exception as e:
            print(f"Skipping item {item.id} due to error: {e}")
            continue
    return stackstac.stack(
        good_items,
        bounds=bounds,
        snap_bounds=False,
        epsg=epsg,
        resolution=gsd,
        dtype="float64",
        rescale=False,
        fill_value=0,
        assets=["blue", "green", "red", "nir"],
        resampling=Resampling.nearest,
    )


# Build stack with only accessible items
stack = safe_stack(items, bounds, epsg, gsd)

print(stack)

# Show progress during download
with ProgressBar():
    stack = stack.compute()

Checking item S2C_43RGP_20251204_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/12/S2C_43RGP_20251204_0_L2A/B04.tif
Checking item S2B_43RGP_20251129_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/11/S2B_43RGP_20251129_0_L2A/B04.tif
Checking item S2A_43RGP_20251126_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/11/S2A_43RGP_20251126_0_L2A/B04.tif
Checking item S2C_43RGP_20251124_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/11/S2C_43RGP_20251124_0_L2A/B04.tif
Checking item S2B_43RGP_20251119_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/11/S2B_43RGP_20251119_0_L2A/B04.tif
Checking item S2C_43RGP_20251114_0_L2A -> https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2025/11/S2C_43RGP_20251114_0_L2A/B04.tif
Checking item S2B_43RGP_20251109_0

RuntimeError: Error opening 'https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2024/3/S2A_43RGP_20240314_0_L2A/B03.tif': RasterioIOError("'/vsicurl/https://sentinel-cogs.s3.us-west-2.amazonaws.com/sentinel-s2-l2a-cogs/43/R/GP/2024/3/S2A_43RGP_20240314_0_L2A/B03.tif' does not exist in the file system, and is not recognized as a supported dataset name.")

In [31]:
import rasterio
import numpy as np
import os


def download_item_to_npz(item, bounds, epsg, gsd, out_dir="cubes"):
    os.makedirs(out_dir, exist_ok=True)
    arrays = []
    band_names = ["blue", "green", "red", "nir"]

    for band in band_names:
        try:
            href = item.assets[band].href
            print(f"Downloading {band} from {item.id} -> {href}")
            with rasterio.open(href) as src:
                # Reproject bounds into dataset CRS
                window = rasterio.windows.from_bounds(*bounds, transform=src.transform)
                arr = src.read(1, window=window, out_shape=(256, 256))
                arrays.append(arr)
        except Exception as e:
            print(f"Skipping {band} in {item.id} due to error: {e}")
            arrays.append(np.zeros((256, 256)))  # placeholder

    cube = np.stack(arrays, axis=-1)  # shape: (256, 256, 4)
    meta = {
        "id": item.id,
        "epsg": epsg,
        "gsd": gsd,
        "bands": band_names,
    }
    np.savez_compressed(os.path.join(out_dir, f"{item.id}.npz"), cube=cube, meta=meta)
    print(f"Saved {item.id}.npz")


# Loop through items
for item in items:
    download_item_to_npz(item, bounds, epsg, gsd)

Saved S2C_43RGP_20251204_0_L2A.npz
Saved S2B_43RGP_20251129_0_L2A.npz
Saved S2A_43RGP_20251126_0_L2A.npz
Saved S2C_43RGP_20251124_0_L2A.npz
Saved S2B_43RGP_20251119_0_L2A.npz
Saved S2C_43RGP_20251114_0_L2A.npz
Saved S2B_43RGP_20251109_0_L2A.npz
Saved S2A_43RGP_20251106_0_L2A.npz
Saved S2C_43RGP_20251104_0_L2A.npz
Skipping blue in S2B_44RKU_20251030_0_L2A due to error: Read failed. See previous exception for details.
Skipping green in S2B_44RKU_20251030_0_L2A due to error: Read failed. See previous exception for details.
Skipping red in S2B_44RKU_20251030_0_L2A due to error: Read failed. See previous exception for details.
Skipping nir in S2B_44RKU_20251030_0_L2A due to error: Read failed. See previous exception for details.
Saved S2B_44RKU_20251030_0_L2A.npz
Saved S2C_43RGP_20251025_0_L2A.npz
Saved S2A_43RGP_20251017_0_L2A.npz
Saved S2C_43RGP_20251015_0_L2A.npz
Saved S2B_43RGP_20251010_0_L2A.npz
Saved S2C_43RGP_20251005_0_L2A.npz
Saved S2B_43RGP_20250930_0_L2A.npz
Saved S2A_43RGP_20250

In [34]:
from pathlib import Path

for npz_file in Path("./cubes").glob("*.npz"):
    sample = np.load(npz_file)
    print(f"{npz_file.name}: keys = {list(sample.keys())}")
    print(sample["meta"])
    # print(f"pixels shape = {sample['meta'].shape}")
    # print(f"week_norm = {sample['week_norm'].shape}")
    break  # just inspect one for now

S2B_43RGP_20240826_0_L2A.npz: keys = ['cube', 'meta']


ValueError: Object arrays cannot be loaded when allow_pickle=False

In [36]:
import numpy as np

sample = np.load("./cubes/S2A_43RGP_20240314_0_L2A.npz", allow_pickle=True)

print("Keys:", sample.files)  # ['cube', 'meta']

cube = sample["cube"]
meta = sample["meta"].item()  # convert object array back to dict

print("Cube shape:", cube.shape)  # e.g. (256, 256, 4)
print("Bands:", meta["bands"])
print("EPSG:", meta["epsg"])
print("GSD:", meta["gsd"])
print("Item ID:", meta["id"])

Keys: ['cube', 'meta']
Cube shape: (256, 256, 4)
Bands: ['blue', 'green', 'red', 'nir']
EPSG: 32643
GSD: 10
Item ID: S2A_43RGP_20240314_0_L2A
